# Auditoria independente das amostras locais da Proposta 1

**Objetivo:** preparar os 2.000 candidatos (250 árvores e 250 gramíneas por região externa) para confirmação por avaliador humano no QGIS.

**Atenção:** a base da Area1 contém 505 segmentos históricos, preservados. Os 2.000 rótulos externos permanecem **sugestões da IA**, não referências definitivas. Os 800 pontos de avaliação geográfica cega são outro conjunto e **não devem ser utilizados para treinamento**.

Este notebook **não calcula F1**, não reclassifica as imagens e não considera qualquer label sugerido pela IA como verdade-terreno. Preserva integralmente arquivos brutos, modelos, resultados históricos e `main`.

## 1. Montar o Google Drive privado

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Obter a branch de experimentos e instalar dependências

In [ ]:
import subprocess, sys
from pathlib import Path

REPO = Path("/content/mestrado-arvores-urbanas-externo")
BRANCH = "feature/validacao-geografica-externa"
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth=1", "--branch", BRANCH,
                    "https://github.com/BrenoBragaGalvao/mestrado-arvores-urbanas.git",
                    str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH], check=True)
SCRIPTS = REPO / "experimentos" / "validacao_externa_regioes"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(SCRIPTS / "requirements.txt")], check=True)
print("Código disponível em", SCRIPTS)

## 3. Descobrir automaticamente os quatro ZIPs originais (sem modificá-los)

In [ ]:
ROOT = Path("/content/drive/MyDrive")
NAMES = [
    "02_sagrada_familia_caxias_250_por_classe.zip",
    "03_centro_novo_hamburgo_250_por_classe.zip",
    "04_lomba_grande_novo_hamburgo_250_por_classe.zip",
    "05_cristo_rei_sao_leopoldo_250_por_classe.zip",
]
found = {}
for p in ROOT.rglob("*_250_por_classe.zip"):
    if p.name in NAMES:
        found.setdefault(p.name, []).append(p)
for name in NAMES:
    options = found.get(name, [])
    print(name, ":", [str(x) for x in options])
    if len(options) != 1:
        raise RuntimeError(f"Esperado exatamente um arquivo {name}, encontrados {len(options)}; ajuste a origem antes de executar.")
PACOTES = [found[name][0] for name in NAMES]

## 4. Produzir GeoPackages cegos e chaves privadas para as quatro regiões

In [ ]:
OUT = ROOT / "Proposta1_REVISAO_CEGA_2026-11-13"
if OUT.exists():
    raise FileExistsError(f"Existe saída anterior em {OUT}. Preserve-a e escolha novo nome para outra execução.")
subprocess.run([sys.executable, str(SCRIPTS / "preparar_revisao_humana_250.py"),
                "preparar", "--pacotes", *map(str, PACOTES), "--out", str(OUT),
                "--epsg", "31982", "--esperadas-por-imagem", "500"], check=True)
print("Pacotes preparados:", OUT)

## 5. Revisão humana e continuidade

Para cada pasta gerada, abra no QGIS o arquivo `amostras_treino_REVISAO_CEGA.gpkg` sobre a **ortoimagem RGB original**. Confira o alinhamento das coordenadas. Não abra `chave_sugestoes_IA_NAO_MOSTRAR_AO_REVISOR.csv` durante a anotação. Marque `classe_verificada` como `arvore`, `grama`, `outro` ou `incerto`, e preencha o nome do avaliador.

Este arquivo de **treinamento** não corresponde aos 200 pontos de teste independentes por região. Após a revisão, a bateria equilibrada deverá ser refeita com os rótulos conferidos, registrando perdas ou correções de classe. Só depois devem ser anotados, separadamente e às cegas, os 800 pontos de teste e calculado o F1-macro de generalização.

Não ajustar hiperparâmetros no conjunto de teste. Não substituir a bateria de 505 segmentos originais da Area1 nem as tabelas históricas. A comparação novo × histórico deve mencionar diferenças de amostragem e protocolo.